# Section 1 – RAG Pipeline & Agentic Workflow (experiments)
Run from the project root folder. Each cell maps to a task in the capstone PDF and produces a required screenshot.

In [ ]:
import os, sys
os.chdir('..') if os.path.basename(os.getcwd()) == 'notebooks' else None
sys.path.append(os.getcwd())

## 1. Data preparation – load, clean, chunk, embed, store in FAISS  *(screenshot: vector store creation)*

In [ ]:
from src.ingestion import build_index
build_index.main()

In [ ]:
import pandas as pd
kb = pd.read_csv('data/kb_rows.csv')
print(kb.intent.value_counts().head(), kb.shape)
kb.head(3)

## 2. Retrieval – semantic vs hybrid (BM25 + FAISS + re-ranker)

In [ ]:
from src.retrieval.retriever import get_retriever
r = get_retriever()
q = 'how can i get my money back for order 123'
for name, docs in [('semantic', r.semantic(q, 4)), ('hybrid', r.hybrid(q, 4))]:
    print(name, [d.metadata['intent'] for d in docs])

## 3. RAG as a tool + structured JSON generation

In [ ]:
from src.tools.rag_tool import rag_answer
rag_answer.invoke({'question': 'What payment methods do you accept?'})

## 4. LangGraph multi-agent workflow  *(screenshot: LangGraph flow diagram)*

In [ ]:
from IPython.display import Image, display
from src.graph.workflow import graph
display(Image(graph.get_graph().draw_mermaid_png()))

## 5. Agent trace + conversational memory  *(screenshot: agent trace output)*

In [ ]:
from src.graph.workflow import run_agentic_rag
import json
for q in ['I want to cancel order 12345, how do I do it?',
          'and will I be charged a fee for that?',
          'this is ridiculous, I was charged twice and nobody helps me! get me a human']:
    print(json.dumps(run_agentic_rag(q, session_id='demo-1'), indent=2)[:1500], '\n')

## 6. Feedback loop – store user feedback, inspect validator verdicts

In [ ]:
from src.tools.feedback import log_user_feedback, feedback_summary
log_user_feedback('demo-1', 'can I pay with UPI?', '(bot answer)', 'down',
                  correction='Yes, UPI is accepted at checkout under {{Online Payment Interaction}}.',
                  intent='check_payment_methods')
feedback_summary()  # corrections are added to the KB on the next build_index run

## 7. Evaluation – Baseline RAG vs Agentic RAG  *(screenshot: evaluation tables)*

In [ ]:
from src.evaluation import evaluate
evaluate.main(n=None)   # use n=27 for a quick run
pd.read_csv('data/eval_summary.csv', index_col=0)